# /geotag — Comprehensive Evaluation

## How /geotag works

```
headline + text
    │
    ├─ Flair NER (ner-spanish-large, F1=90.54)
    │   └─ LOC spans with char offsets
    │
    ├─ Street regex post-pass
    │   └─ Prefixed spans (Calle X, Avenida X, Plaza X…) not already covered by NER
    │
    ├─ Street rescue
    │   └─ NER LOC spans with street prefix + no gazetteer hit → moved to street pool
    │
    ├─ Gazetteer lookup (geonames_es.tsv)
    │   ├─ P-class entries → city candidates
    │   └─ A-class entries → region candidates
    │
    ├─ NLI Stage 1  (Recognai XNLI)
    │   └─ Scope: local / regional / national
    │
    ├─ NLI Stage 2  (conditional)
    │   ├─ local  → pick winning city from P-class pool
    │   └─ regional → pick winning region from A-class pool
    │
    └─ Street resolver
        └─ edge_ids from street_index per city
```

**Known limitations:**
- Flair may extract "Ayuntamiento de Madrid" as ORG → Madrid not in LOC output
- Compound spans like "Gran Vía de Madrid" absorb the city name
- "La Rambla" maps to a Córdoba municipality in GeoNames, not Barcelona's landmark
- Districts (Eixample, Gracia) only resolve to parent city if the city also appears explicitly

**Sections:**
1. Span Detection — does Flair extract the expected named entities?
2. Span Classification — are street / city / region types correct?
3. City Resolution — does the right city win from the candidate pool?
4. Scope Classification — local / regional / national?
5. Scope → /classify Passthrough — does the scope flow correctly downstream?

**Prerequisite:** NLP service running. `/readyz` → 200.

In [23]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('geotag_cases.json')
by_section = {}
for c in cases:
    by_section.setdefault(c['section'], []).append(c)
print(f'Loaded {len(cases)} cases across sections: {list(by_section.keys())}')

Loaded 15 cases across sections: ['span_detection', 'city_resolution', 'scope']


In [24]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [26]:
def chk(ok, known_issue=False):
    if ok:          return '✓'
    if known_issue: return '⚠'
    return '✗'


def call_geotag(case):
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text':       case['text'],
            'headline':   case.get('headline', ''),
            'source':     case.get('source', ''),
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    return resp.json(), latency


def spans_in_places(expected_text, all_places):
    """True if any place text contains expected_text (case-insensitive)."""
    low = expected_text.lower()
    return any(low in p['text'].lower() for p in all_places)


def print_header(section_name, description):
    print(f'\n{"═" * 72}')
    print(f'  {section_name}: {description}')
    print(f'{"═" * 72}')

## Section 1 — Span Detection

Checks that Flair NER + street regex extract the expected text spans into `all_places`.
A ⚠ means the span is missing due to a **known model limitation** (documented in the fixture).
A ✗ means an unexpected regression.

In [27]:
s1_results = []

for case in by_section.get('span_detection', []):
    data, latency = call_geotag(case)
    all_places = data.get('all_places', [])
    all_texts = [p['text'] for p in all_places]

    expected = case.get('expected_spans_contain', [])
    span_checks = []
    for exp in expected:
        found = spans_in_places(exp, all_places)
        span_checks.append((exp, found))

    all_ok = all(ok for _, ok in span_checks)
    has_issue = bool(case.get('known_issues'))

    print(f'  [{case["id"]}]  {case["description"]}')
    print(f'    text : {case["text"][:100]}...' if len(case['text']) > 100 else f'    text : {case["text"]}')
    print(f'    detected places : {all_texts}')
    for exp, ok in span_checks:
        icon = chk(ok, known_issue=(not ok and has_issue))
        note = '  ← KNOWN ISSUE' if not ok and has_issue else ''
        print(f'    {icon}  span {exp!r}{note}')
    if not all_ok and case.get('notes'):
        print(f'    note: {case["notes"]}')
    print(f'    latency: {latency:.2f}s')
    print()

    passed = all_ok
    s1_results.append({'id': case['id'], 'pass': passed, 'known': has_issue and not passed})

hard_fails = [r for r in s1_results if not r['pass'] and not r['known']]
warns      = [r for r in s1_results if r['known']]
passed_s1  = [r for r in s1_results if r['pass']]
print(f'Span Detection:  {len(passed_s1)}/{len(s1_results)} pass  '
      f'| {len(warns)} known issues  | {len(hard_fails)} unexpected failures')

  [geo-sd01]  City name appears standalone in body — Flair should extract it directly
    text : El Ayuntamiento de Valladolid aprobó ayer un plan de 3 millones de euros para ampliar la red de carr...
    detected places : ['Valladolid']
    ✓  span 'Valladolid'
    latency: 0.51s

  [geo-sd02]  City inside 'Ayuntamiento de X' — Flair may tag whole phrase as ORG not LOC
    text : El Ayuntamiento de Madrid ha aprobado la ampliación del carril bici en la Gran Vía. Las obras durará...
    detected places : ['Gran Vía', 'Gran Vía']
    ⚠  span 'Madrid'  ← KNOWN ISSUE
    note: Flair may extract 'Gran Vía de Madrid' as a compound LOC instead of separating Madrid and Gran Vía
    latency: 0.24s

  [geo-sd03]  Street prefix in text — regex should produce a street span
    text : El Ayuntamiento de Madrid ha comunicado el corte de la Calle Serrano entre Velázquez y Goya durante ...
    detected places : ['Velázquez', 'Goya', 'Calle Serrano', 'Calle Serrano']
    ✓  span 'Calle Serrano'
    la

## Section 2 — Span Classification

Verifies that detected spans have the correct type in `all_places` (city / region / street / other)
and appear in the appropriate response field (`geo_cities`, `geo_streets`).
Also checks that `all_places` has no duplicate spans.

In [28]:
all_cases_for_s2 = cases  # run on all cases
s2_results = []

for case in all_cases_for_s2:
    data, latency = call_geotag(case)
    all_places  = data.get('all_places', [])
    geo_cities  = data.get('geo_cities', [])
    geo_streets = data.get('geo_streets', [])
    geo_points  = data.get('geo_points', [])

    # Duplicates check
    seen_texts = [p['text'] for p in all_places]
    has_dupes  = len(seen_texts) != len(set(seen_texts))

    # Type breakdown
    type_counts = {}
    for p in all_places:
        type_counts[p['type']] = type_counts.get(p['type'], 0) + 1

    # Streets in geo_streets vs streets in all_places
    places_streets = [p['text'] for p in all_places if p['type'] == 'street']
    resolver_spans = [s['span'] for s in geo_streets]

    print(f'  [{case["id"]}]  {case["description"]}')
    print(f'    all_places types : {dict(type_counts)}')
    if geo_cities:
        print(f'    geo_cities       : {[c["city_name"] for c in geo_cities]}')
    if geo_streets:
        print(f'    geo_streets      : {resolver_spans}')
    if geo_points:
        print(f'    geo_points       : {[p["span"] for p in geo_points]}')
    dupe_icon = '✗' if has_dupes else '✓'
    print(f'    {dupe_icon}  no duplicate spans in all_places'
          + (f'  DUPES: {[t for t in seen_texts if seen_texts.count(t)>1]}' if has_dupes else ''))
    if case.get('expected_streets_min', 0) > 0:
        street_ok = len(geo_streets) >= case['expected_streets_min']
        print(f'    {chk(street_ok)}  streets: got {len(geo_streets)}  min {case["expected_streets_min"]}')
    print(f'    latency: {latency:.2f}s')
    print()

    s2_results.append({
        'id': case['id'],
        'no_dupes': not has_dupes,
        'streets_ok': len(geo_streets) >= case.get('expected_streets_min', 0),
    })

dupe_ok   = sum(1 for r in s2_results if r['no_dupes'])
str_ok    = sum(1 for r in s2_results if r['streets_ok'])
print(f'Span Classification:  no-duplicate check {dupe_ok}/{len(s2_results)}  '
      f'| street-count check {str_ok}/{len(s2_results)}')

  [geo-sd01]  City name appears standalone in body — Flair should extract it directly
    all_places types : {'region': 1}
    geo_points       : ['Valladolid']
    ✓  no duplicate spans in all_places
    latency: 0.27s

  [geo-sd02]  City inside 'Ayuntamiento de X' — Flair may tag whole phrase as ORG not LOC
    all_places types : {'other': 2}
    ✗  no duplicate spans in all_places  DUPES: ['Gran Vía', 'Gran Vía']
    latency: 0.26s

  [geo-sd03]  Street prefix in text — regex should produce a street span
    all_places types : {'other': 1, 'city': 1, 'street': 2}
    geo_streets      : ['Calle Serrano', 'Calle Serrano']
    geo_points       : ['Goya']
    ✗  no duplicate spans in all_places  DUPES: ['Calle Serrano', 'Calle Serrano']
    ✓  streets: got 2  min 1
    latency: 0.27s

  [geo-sd04]  Multiple street spans in one article
    all_places types : {'street': 4}
    geo_streets      : ['Calle Alcalá', 'Calle Alcalá', 'Plaza de Cibeles', 'Calle Goya']
    ✗  no duplicate spans i

## Section 3 — City Resolution

Checks that the winning city in `geo_cities` matches the expected city.
Shows NLI confidence and what other candidates were available.

In [29]:
s3_cases = by_section.get('city_resolution', []) + by_section.get('scope', [])
s3_results = []

for case in s3_cases:
    if not case.get('expected_city_present', True) and case.get('expected_city') is None:
        continue  # skip cases where no city is expected

    data, latency = call_geotag(case)
    geo_cities  = data.get('geo_cities', [])
    all_places  = data.get('all_places', [])
    top_city    = geo_cities[0]['city_name'] if geo_cities else None
    confidence  = geo_cities[0]['confidence'] if geo_cities else 0.0

    exp_city    = case.get('expected_city')
    exp_present = case.get('expected_city_present', True)
    city_ok     = (top_city == exp_city) if exp_present else (top_city is None)

    all_city_places = [p['text'] for p in all_places if p['type'] in ('city', 'region')]
    has_issue = bool(case.get('known_issues'))
    icon = chk(city_ok, known_issue=not city_ok and has_issue)

    print(f'  {icon} [{case["id"]}]  {case["description"]}')
    print(f'    expected city  : {exp_city!r}  (present={exp_present})')
    print(f'    resolved city  : {top_city!r}  confidence={confidence:.3f}')
    print(f'    city candidates in places: {all_city_places}')
    if not city_ok and case.get('notes'):
        print(f'    note: {case["notes"]}')
    print(f'    latency: {latency:.2f}s')
    print()

    s3_results.append({'id': case['id'], 'pass': city_ok, 'known': has_issue and not city_ok})

passed_s3 = sum(1 for r in s3_results if r['pass'])
warns_s3  = sum(1 for r in s3_results if r['known'])
print(f'City Resolution:  {passed_s3}/{len(s3_results)} correct  | {warns_s3} known issues')

  ✗ [geo-cr01]  Single clear city mention in body
    expected city  : 'Sevilla'  (present=True)
    resolved city  : None  confidence=0.000
    city candidates in places: ['Sevilla']
    note: City appears only in headline (BiciSevilla); Flair must detect it from the combined headline+text input
    latency: 0.25s

  ✓ [geo-cr02]  District resolves to parent city — Gracia → Barcelona
    expected city  : 'Barcelona'  (present=True)
    resolved city  : 'Barcelona'  confidence=1.000
    city candidates in places: ['Gracia', 'Gracia', 'Barcelona']
    latency: 0.25s

  ✗ [geo-cr03]  Multiple city comparison — dominant city (Barcelona) should win over mentioned cities
    expected city  : 'Barcelona'  (present=True)
    resolved city  : None  confidence=0.000
    city candidates in places: ['Barcelona', 'Madrid', 'Barcelona', 'Madrid', 'Sevilla']
    note: Barcelona appears first and is the article subject; NLI should prefer it over Madrid (higher population) and Sevilla
    latency: 0.2

## Section 4 — Scope Classification

Checks `geo_scope` against the expected scope (city / regional / national).
Shows the NLI-driven decision for each case.

In [30]:
s4_results = []

for case in cases:
    if not case.get('expected_geo_scope'):
        continue

    data, latency = call_geotag(case)
    geo_scope    = data['geo_scope']
    exp_scope    = case['expected_geo_scope']
    scope_ok     = geo_scope == exp_scope
    has_issue    = bool(case.get('known_issues'))
    icon         = chk(scope_ok, known_issue=not scope_ok and has_issue)

    print(f'  {icon} [{case["id"]}]  {case["description"]}')
    print(f'    expected scope : {exp_scope!r}')
    print(f'    got scope      : {geo_scope!r}')
    if not scope_ok and case.get('notes'):
        print(f'    note: {case["notes"]}')
    print(f'    latency: {latency:.2f}s')
    print()

    s4_results.append({'id': case['id'], 'pass': scope_ok, 'known': has_issue and not scope_ok})

passed_s4 = sum(1 for r in s4_results if r['pass'])
warns_s4  = sum(1 for r in s4_results if r['known'])
print(f'Scope:  {passed_s4}/{len(s4_results)} correct  | {warns_s4} known issues')

  ✓ [geo-sd01]  City name appears standalone in body — Flair should extract it directly
    expected scope : 'city'
    got scope      : 'city'
    latency: 0.27s

  ✓ [geo-sd02]  City inside 'Ayuntamiento de X' — Flair may tag whole phrase as ORG not LOC
    expected scope : 'city'
    got scope      : 'city'
    latency: 0.24s

  ✓ [geo-sd03]  Street prefix in text — regex should produce a street span
    expected scope : 'city'
    got scope      : 'city'
    latency: 0.24s

  ✓ [geo-sd04]  Multiple street spans in one article
    expected scope : 'city'
    got scope      : 'city'
    latency: 0.25s

  ✓ [geo-sd05]  District name — Eixample should appear in spans, Barcelona as city
    expected scope : 'city'
    got scope      : 'city'
    latency: 0.29s

  ✗ [geo-sd06]  Multiple city spans — all should appear in detected places
    expected scope : 'city'
    got scope      : 'regional'
    note: Barcelona is the article subject; NLI should pick it over Madrid despite Madrid havi

## Section 5 — Scope → /classify Passthrough

For each case:
1. Calls `/geotag` to get `geo_scope`
2. Passes that scope to `/classify` (scope NLI is skipped; returned scope must match exactly)
3. Shows topics and out_of_scope verdict

This verifies that the geo_scope flows correctly from the geotagger into the classifier.

In [33]:
s5_results = []

for case in cases:
    data, _ = call_geotag(case)
    geo_scope = data['geo_scope']

    resp_cls = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={
            'article_id':  case['article_id'],
            'summary':     case['text'],
            'geo_cities':  data.get('geo_cities', []),
            'search_tags': [],
            'geo_scope':   geo_scope,
        },
        headers=HEADERS,
    )
    if resp_cls.status_code != 200:
        print(f'  ❌ [{case["article_id"]}]  /classify HTTP {resp_cls.status_code}')
        s5_results.append(False)
        continue

    cls = resp_cls.json()
    scope_match = cls['geo_scope'] == geo_scope
    icon = '✓' if scope_match else '✗'
    oos  = cls['out_of_scope']

    s5_results.append(scope_match)
    print(f'  {icon} [{case["article_id"]}]  '
          f'geotag={geo_scope!r} → classify={cls["geo_scope"]!r}  '
          f'oos={oos}  topics={cls["topics"]}')

print()
print(f'Scope passthrough: {sum(s5_results)}/{len(s5_results)} match')

  ✓ [geo-sd01]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sd02]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sd03]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sd04]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sd05]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sd06]  geotag='regional' → classify='regional'  oos=True  topics=[]
  ✓ [geo-cr01]  geotag='regional' → classify='regional'  oos=True  topics=[]
  ✓ [geo-cr02]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-cr03]  geotag='regional' → classify='regional'  oos=False  topics=['movilidad sostenible', 'infraestructura ciclista', 'evento de movilidad']
  ✓ [geo-cr04]  geotag='city' → classify='city'  oos=True  topics=[]
  ✓ [geo-sc01]  geotag='city' → classify='city'  oos=False  topics=['infraestructura ciclista', 'movilidad sostenible', 'aparcamiento de bicicletas']
  ✓ [geo-sc02]  geotag='regional' → classify='regional'  oos=T

## Overall Scorecard

In [35]:
print_scorecard('/geotag comprehensive', {
    'Total cases':                    len(cases),
    'S1 Span detection (pass)':       f'{len(passed_s1)}/{len(s1_results)}',
    'S1 Known issues (⚠)':            len(warns_s1),
    'S1 Unexpected failures (✗)':     len(hard_fails),
    'S2 No-duplicate check':          f'{dupe_ok}/{len(s2_results)}',
    'S2 Street count correct':        f'{str_ok}/{len(s2_results)}',
    'S3 City resolution correct':     f'{passed_s3}/{len(s3_results)}',
    'S4 Scope correct':               f'{passed_s4}/{len(s4_results)}',
    'S5 Scope passthrough':           f'{sum(s5_results)}/{len(s5_results)}',
})

NameError: name 'warns_s1' is not defined